In [3]:
import pandas as pd
import glob
import os
import folium

In [4]:
data_folder = './Opencellid' 
columns = ['radio', 'mcc', 'net', 'area', 'cell', 'unit', 'lon', 'lat', 
           'range', 'samples', 'changeable', 'created', 'updated', 'averageSignal']

In [5]:
# combine all the 3 files into one dataframe
all_files = glob.glob(os.path.join(data_folder, "*.csv"))
df_list = []
for file in all_files:
    df = pd.read_csv(file, header=None, names=columns)
    df_list.append(df)
if df_list:
    topology_df = pd.concat(df_list, ignore_index=True)
else:
    print("No CSV files found. Check your data_folder path and file names.")
    topology_df = pd.DataFrame()

#export the combined dataframe to a new csv file
# write to a temp file then atomically replace to avoid permission issues
tmp_path = os.path.join(data_folder, f"combined_topology.tmp.{os.getpid()}.csv")
topology_df.to_csv(tmp_path, index=False)
try:
    os.replace(tmp_path, os.path.join(data_folder, "combined_topology.csv"))
except PermissionError:
    print(f"Permission denied writing combined_topology.csv; temp file left at {tmp_path}")

C:\Users\HP\AppData\Local\Temp\ipykernel_48068\1420345029.py:5: DtypeWarning: Columns (0: mcc, 1: net, 2: area, 3: cell, 4: unit, 5: lon, 6: lat, 7: range, 8: samples, 9: changeable, 10: created, 11: updated, 12: averageSignal) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, header=None, names=columns)


In [6]:
# import the combined csv file into a new dataframe
topology_df = pd.read_csv(os.path.join(data_folder, "combined_topology.csv"))

# Filter for LTE and London/UCL area
lte_df = topology_df[topology_df['radio'] == 'LTE'].copy()

lte_df['lat'] = pd.to_numeric(lte_df['lat'], errors='coerce')
lte_df['lon'] = pd.to_numeric(lte_df['lon'], errors='coerce')
lte_df = lte_df.dropna(subset=['lat', 'lon'])

# Just adjust these values in your existing cell and re-run
lat_min, lat_max = 51.522430, 51.52679  
lon_min, lon_max = -0.136141, -0.126012  
# Re-filter
topology_df = lte_df[
    (lte_df['lat'] >= lat_min) & (lte_df['lat'] <= lat_max) &
    (lte_df['lon'] >= lon_min) & (lte_df['lon'] <= lon_max)
].copy()

topology_df = topology_df.drop_duplicates(subset=['lat', 'lon']).copy()

print(f"Number of LTE sites in your selected area: {len(topology_df)}")

C:\Users\HP\AppData\Local\Temp\ipykernel_48068\3226216334.py:2: DtypeWarning: Columns (0: mcc, 1: net, 2: area, 3: cell, 4: unit, 5: lon, 6: lat, 7: range, 8: samples, 9: changeable, 10: created, 11: updated, 12: averageSignal) have mixed types. Specify dtype option on import or set low_memory=False.
  topology_df = pd.read_csv(os.path.join(data_folder, "combined_topology.csv"))


Number of LTE sites in your selected area: 46


In [7]:
# 1. Initialize the map centered on your bounding box
center_lat = (lat_min + lat_max) / 2
center_lon = (lon_min + lon_max) / 2
m = folium.Map(location=[center_lat, center_lon], zoom_start=16, tiles='cartodbpositron')

# 2. Add the bounding box visual
folium.Rectangle(
    bounds=[[lat_min, lon_min], [lat_max, lon_max]],
    color='red',
    fill=True,
    fill_opacity=0.05,
    popup="Selected Topology Area"
).add_to(m)

# 3. Add markers for each cell tower
for idx, row in topology_df.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=5,
        color='blue',
        fill=True,
        fill_color='blue',
        popup=f"Cell ID: {row['cell']}<br>Lat: {row['lat']}<br>Lon: {row['lon']}"
    ).add_to(m)

# 4. Save and display
m.save("topology_map.html") # Saves to a file you can open in your browser
m

In [8]:
# Step 1: load raw
raw_df = pd.read_csv(os.path.join(data_folder, "combined_topology.csv"))

# Step 2: filter LTE
lte_df = raw_df[raw_df['radio'] == 'LTE'].copy()

# Step 3: force numeric
lte_df['lat'] = pd.to_numeric(lte_df['lat'], errors='coerce')
lte_df['lon'] = pd.to_numeric(lte_df['lon'], errors='coerce')
lte_df = lte_df.dropna(subset=['lat', 'lon'])

# Step 4: box filter — give it its own distinct name
lat_min, lat_max = 51.522430, 51.52679
lon_min, lon_max = -0.136141, -0.126012

ucl_box_df = lte_df[
    (lte_df['lat'] >= lat_min) & (lte_df['lat'] <= lat_max) &
    (lte_df['lon'] >= lon_min) & (lte_df['lon'] <= lon_max)
].copy()

print(f"Rows in UCL box: {len(ucl_box_df)}")
print(f"Unique lat/lon pairs in UCL box: {ucl_box_df[['lat','lon']].drop_duplicates().shape[0]}")

C:\Users\HP\AppData\Local\Temp\ipykernel_48068\1008586175.py:2: DtypeWarning: Columns (0: mcc, 1: net, 2: area, 3: cell, 4: unit, 5: lon, 6: lat, 7: range, 8: samples, 9: changeable, 10: created, 11: updated, 12: averageSignal) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv(os.path.join(data_folder, "combined_topology.csv"))


Rows in UCL box: 230
Unique lat/lon pairs in UCL box: 46


In [9]:
# export only the filtered dataframe to a new csv file
topology_df.to_csv(os.path.join(data_folder, "only_UCL_cells.csv"), index=False)


In [10]:
# import the filtered csv file into a new dataframe
filtered_topology_df = pd.read_csv(os.path.join(data_folder, "only_UCL_cells.csv"))

In [11]:
#drop all ther columns except MCC, MNC, LAC, cell ID, lat and lon
filtered_topology_df = filtered_topology_df[['mcc', 'net', 'area', 'cell', 'lat', 'lon']].copy()
# rename the columns to MCC, MNC, LAC, CellID, Latitude and Longitude
filtered_topology_df.columns = ['MCC', 'MNC', 'LAC', 'CellID', 'Latitude', 'Longitude']

#create new column macro and set it to 1 for all rows
filtered_topology_df['Macro'] = 1

In [12]:
# plot the points on a map using folium and label with the cell ID here not html
m = folium.Map(location=[center_lat, center_lon], zoom_start=16, tiles='cartodbpositron')
for idx, row in filtered_topology_df.iterrows():
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=5,
        color='blue',
        fill=True,
        fill_color='blue',
        popup=f"Cell ID: {row['CellID']}<br>Lat: {row['Latitude']}<br>Lon: {row['Longitude']}"
    ).add_to(m)
m.save("filtered_topology_map.html") # Saves to a file you can open in your browser
m    


In [13]:
# add a column called "Macro" and set it to 1 for all rows
filtered_topology_df['Macro'] = 1

In [14]:
#convert the cell IDs to strings
filtered_topology_df['CellID'] = filtered_topology_df['CellID'].astype(str)
filtered_topology_df['Macro'] = filtered_topology_df['Macro'].astype(str)

In [15]:
filtered_topology_df.head()

,MCC,MNC,LAC,CellID,Latitude,Longitude,Macro
0,234,20,1003,3304449,51.5266,-0.1338,1
1,234,15,6187,2176530,51.5248,-0.1359,1
2,234,15,6188,291362,51.5250,-0.1275,1
3,234,10,14353,136015224,51.5260,-0.1345,1
4,234,10,14353,129595522,51.5255,-0.1327,1


In [16]:
# for the cells with the following cell IDs: 136015219.0, 2059042.0, 153801080.0, 4178450.0, 137283710.0, 2601987.0, 2602003.0, Rename their cell IDs to M1, M2... and set the Macro column to their same cell id for these cells
cell_ids = ['136015219', '2059042', '153801080', '4178450', '137283710', '2601987', '2602003']
macro_values = ['M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7']

for i, cell_id in enumerate(cell_ids):
    if cell_id in filtered_topology_df['CellID'].values:
        filtered_topology_df.loc[filtered_topology_df['CellID'] == cell_id, 'CellID'] = macro_values[i]
        filtered_topology_df.loc[filtered_topology_df['CellID'] == macro_values[i], 'Macro'] = macro_values[i]
    else:
        print(f"Cell ID {cell_id} not found in the dataframe.")    


In [17]:
filtered_topology_df

,MCC,MNC,LAC,CellID,Latitude,Longitude,Macro
0,234,20,1003,3304449,51.5266,-0.1338,1
1,234,15,6187,2176530,51.5248,-0.1359,1
2,234,15,6188,291362,51.5250,-0.1275,1
3,234,10,14353,136015224,51.5260,-0.1345,1
4,234,10,14353,129595522,51.5255,-0.1327,1
5,234,10,14353,136846962,51.5228,-0.1322,1
6,234,10,14353,135691637,51.5250,-0.1312,1
7,234,10,14353,M5,51.5263,-0.1285,M5
8,234,10,14353,137283711,51.5260,-0.1276,1
9,234,10,14353,135372402,51.5262,-0.1337,1


In [18]:
# for all the other cells, name them C1, C2... and set the Macro column to their closest macro cell based on the distance between their lat and long coordinates
from geopy.distance import geodesic
macro_cells = filtered_topology_df[filtered_topology_df['Macro'] != '1'].copy()
other_cells = filtered_topology_df[filtered_topology_df['Macro'] == '1'].copy()
for idx, row in other_cells.iterrows():
    cell_location = (row['Latitude'], row['Longitude'])
    closest_macro = None
    min_distance = float('inf')
    
    for _, macro_row in macro_cells.iterrows():
        macro_location = (macro_row['Latitude'], macro_row['Longitude'])
        distance = geodesic(cell_location, macro_location).meters
        
        if distance < min_distance:
            min_distance = distance
            closest_macro = macro_row['Macro']
    
    filtered_topology_df.loc[idx, 'Macro'] = closest_macro
    filtered_topology_df.loc[idx, 'CellID'] = f"C{idx+1}"  # Name them C1, C2... based on their index

    


In [19]:
filtered_topology_df

,MCC,MNC,LAC,CellID,Latitude,Longitude,Macro
0,234,20,1003,C1,51.5266,-0.1338,M3
1,234,15,6187,C2,51.5248,-0.1359,M1
2,234,15,6188,C3,51.5250,-0.1275,M6
3,234,10,14353,C4,51.5260,-0.1345,M3
4,234,10,14353,C5,51.5255,-0.1327,M4
5,234,10,14353,C6,51.5228,-0.1322,M2
6,234,10,14353,C7,51.5250,-0.1312,M4
7,234,10,14353,M5,51.5263,-0.1285,M5
8,234,10,14353,C9,51.5260,-0.1276,M5
9,234,10,14353,C10,51.5262,-0.1337,M3


In [20]:
# plot the cell towers on a map using folium, all the cells with the same macro cell should have the same color
m = folium.Map(location=[center_lat, center_lon], zoom_start=16, tiles='cartodbpositron')
macro_colors = {'M1': 'red', 'M2': 'green', 'M3': 'blue', 'M4': 'purple', 'M5': 'orange', 'M6': 'darkred', 'M7': 'darkblue'}
for idx, row in filtered_topology_df.iterrows():
    color = macro_colors.get(row['Macro'], 'gray')  # Default to gray if macro cell is not in the dictionary
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=5,
        color=color,
        fill=True,
        fill_color=color,
        popup=f"Cell ID: {row['CellID']}<br>Macro Cell: {row['Macro']}<br>Lat: {row['Latitude']}<br>Lon: {row['Longitude']}"
    ).add_to(m)
m.save("final_topology_map.html") # Saves to a file you can open in your browser
m

In [21]:
# remmove first 3 columns from the filtered_topology_df 
final_topology_df = filtered_topology_df.drop(columns=['MCC', 'MNC', 'LAC'])
final_topology_df

,CellID,Latitude,Longitude,Macro
0,C1,51.5266,-0.1338,M3
1,C2,51.5248,-0.1359,M1
2,C3,51.5250,-0.1275,M6
3,C4,51.5260,-0.1345,M3
4,C5,51.5255,-0.1327,M4
5,C6,51.5228,-0.1322,M2
6,C7,51.5250,-0.1312,M4
7,M5,51.5263,-0.1285,M5
8,C9,51.5260,-0.1276,M5
9,C10,51.5262,-0.1337,M3


In [25]:
# plot the cell towers on a map using folium
# macro cells themselves are black; micro cells are colored by which macro they belong to
import folium
m = folium.Map(location=[center_lat, center_lon], zoom_start=16, tiles='cartodbpositron')
macro_colors = {'M1': 'red', 'M2': 'green', 'M3': 'blue', 'M4': 'purple', 'M5': 'orange', 'M6': 'darkred', 'M7': 'darkblue'}

for idx, row in filtered_topology_df.iterrows():
    if str(row['CellID']).startswith('M'):
        color = 'black'
    else:
        color = macro_colors.get(row['Macro'], 'gray')  # Default to gray if macro cell is not in the dictionary
    
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=5,
        color=color,
        fill=True,
        fill_color=color,
        popup=f"Cell ID: {row['CellID']}<br>Macro Cell: {row['Macro']}<br>Lat: {row['Latitude']}<br>Lon: {row['Longitude']}"
    ).add_to(m)

m.save("final_topology_map.html")
m

In [ ]:
# export the final dataframe to a new csv file
final_topology_df.to_csv(os.path.join(data_folder, "final_topology.csv"), index=False)